# 02 · Explore price variation and demand

**Question:** Which demand patterns should the elasticity model account for?

**Input:** `data/processed/cereal_analysis.parquet` from notebook 01.
This notebook summarizes product performance, then examines Cheerios 15 oz (UPC `1600066610`), the original analysis's highest-revenue product. Charts are descriptive associations, not estimates of a causal price response. No files are exported.

## 1. Load the analysis sample

The first notebook has already applied the sample rules. Keep that sample unchanged here so exploration and modeling use the same records.

In [ ]:
from pathlib import Path

# Support kernels started in the repository root or notebooks directory.
PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
if not (PROJECT / "notebooks").is_dir() or not (PROJECT / "README.md").is_file():
    raise RuntimeError("Start the notebook from the repository root or notebooks directory.")

import pandas as pd
import matplotlib.pyplot as plt

analysis = pd.read_parquet(
    PROJECT / "data" / "processed" / "cereal_analysis.parquet"
)

print(f"Loaded {len(analysis):,} analysis records.")

## 2. Identify important products and usable price variation

Rank products by revenue before estimating elasticity. Combine volume, revenue, coverage, and price variation in one table instead of repeatedly printing overlapping summaries. `price_cv` is the price standard deviation divided by its mean; larger values indicate greater relative dispersion. The mean price is an unweighted average across source records, not revenue divided by units.

In [ ]:
product_summary = (
    analysis.groupby(["UPC", "DESCRIP", "SIZE"])
    .agg(
        total_units=("MOVE", "sum"), total_revenue=("REVENUE", "sum"),
        observations=("MOVE", "size"), stores=("STORE", "nunique"),
        weeks=("WEEK", "nunique"), avg_price=("UNIT_PRICE", "mean"),
        min_price=("UNIT_PRICE", "min"), max_price=("UNIT_PRICE", "max"),
        price_std=("UNIT_PRICE", "std"), unique_prices=("UNIT_PRICE", "nunique"),
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)
product_summary["price_range"] = product_summary["max_price"] - product_summary["min_price"]
product_summary["price_cv"] = product_summary["price_std"] / product_summary["avg_price"]
top_share = product_summary.head(10)["total_revenue"].sum() / product_summary["total_revenue"].sum()
print(f"Top 10 products' share of revenue: {top_share:.2%}")
product_summary.head(10)

## 3. Compare weekly price and sales patterns

Weekly plots show whether prices and volumes move together over time. Average prices are unweighted record means; volumes are weekly totals. Changes in participating stores can also change total volume, so these plots alone do not identify a price effect.

In [ ]:
cheerios = analysis.loc[analysis["UPC"] == 1600066610].copy()
cheerios_weekly = (
    cheerios.groupby("WEEK")
    .agg(avg_price=("UNIT_PRICE", "mean"), total_units=("MOVE", "sum"))
    .reset_index()
)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(cheerios_weekly["WEEK"], cheerios_weekly["avg_price"])
axes[0].set_ylabel("Average unit price ($)")
axes[0].set_title("Cheerios 15 oz: weekly price and unit sales")
axes[1].plot(cheerios_weekly["WEEK"], cheerios_weekly["total_units"])
axes[1].set(xlabel="Week", ylabel="Units sold")
plt.tight_layout()
plt.show()

## 4. Examine the unadjusted price–quantity relationship

Each point averages unit sales across records at one price. Points are not weighted by their number of observations. A downward pattern may reflect price response, but promotions, store differences, and timing can also produce it; the next checks motivate controls for those factors.

In [ ]:
cheerios_price_demand = (
    cheerios.groupby("UNIT_PRICE")
    .agg(
        avg_units=("MOVE", "mean"),
        observations=("MOVE", "size")
    )
    .reset_index()
)

plt.figure(figsize=(8, 6))

plt.scatter(
    cheerios_price_demand["UNIT_PRICE"],
    cheerios_price_demand["avg_units"]
)

plt.xlabel("Unit Price ($)")
plt.ylabel("Average Units Sold")
plt.title("Cheerios 15 oz — Price vs. Average Unit Sales")

plt.show()

## 5. Compare recorded promotion categories

Use one category table to compare average prices, average and median unit sales, and revenue. Missing codes mean **no code recorded**, not proof that no promotion occurred. Category differences motivate the categorical promotion controls in the Cheerios model; undocumented labels should not be assigned stronger meanings.

In [ ]:
promo_type_summary = (
    cheerios.assign(
        SALE_CODE=cheerios["SALE"].fillna("No code recorded")
    )
    .groupby("SALE_CODE")
    .agg(
        observations=("MOVE", "size"),
        avg_price=("UNIT_PRICE", "mean"),
        avg_units=("MOVE", "mean"),
        median_units=("MOVE", "median"),
        avg_revenue=("REVENUE", "mean")
    )
    .sort_values("observations", ascending=False)
)

promo_type_summary

## 6. Check variation within stores and within weeks

Store-level averages describe persistent differences in demand. Within-store price variation helps distinguish changing prices from fixed store characteristics. Within-week variation checks whether prices differ across stores even after common timing effects are absorbed. Zero or very limited variation would weaken the information available for estimating the price coefficient.

In [ ]:
store_summary = cheerios.groupby("STORE").agg(
    avg_units=("MOVE", "mean"), price_std=("UNIT_PRICE", "std"),
    unique_prices=("UNIT_PRICE", "nunique"),
)
week_summary = cheerios.groupby("WEEK").agg(
    stores=("STORE", "nunique"), price_std=("UNIT_PRICE", "std"),
    unique_prices=("UNIT_PRICE", "nunique"),
)
print("Across stores:\n", store_summary.describe().round(2).to_string())
print("\nAcross weeks:\n", week_summary.describe().round(2).to_string())

## What this means for modeling

The original analysis found differences across stores, recorded promotion categories, and weeks. Notebook 03 therefore compares a price-only model with models that add promotion, store, and week controls. Read the tables above to check whether that rationale still holds when using new data.

These controls reduce specific sources of confounding, but prices were not randomly assigned. The estimated elasticity remains a conditional association.